# Figure 2: perturbation programs across sequencing- and imaging-based screens

Run this notebook from the `figures/` folder after populating `data/` and `results/`
(see the repository README). Each section below reruns the relevant part of an analysis
notebook from the saved models; panels are written to `figures/panels/`
(`paths.FIGURES_DIR`).

In [ ]:
import os
import sys

# Repository root, so that paths.py (DATA_DIR / RESULTS_DIR / FIGURES_DIR) can be imported.
sys.path.insert(0, os.path.abspath('..'))
import paths

for d in (paths.DATA_DIR, paths.RESULTS_DIR, paths.FIGURES_DIR):
    print(d, '(found)' if os.path.isdir(d) else '(missing)')

## Spatial screen (Perturb-RAEFISH)

From [`scripts/raefish/2_raefish_analysis.ipynb`](../scripts/raefish/2_raefish_analysis.ipynb).

The GO enrichment cells query Enrichr through `gseapy` and need internet access.

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import sys

import seaborn as sns
import matplotlib.pyplot as plt
import random

import torch

import sherlock as slk

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
slk.configs.show()
slk.configs.set_config('ntc_label', 'non-targeting')
print('\n')
slk.configs.show()
slk.configs.reset_config()
slk.configs.get_config('ntc_label')

In [ ]:
data = sc.read_h5ad(f"{paths.DATA_DIR}/raefish/raefish.h5ad")
slk.pp.slk_prepare_data(data, pert_key="RaeFishGene", ntc_label="NonTargeting", treatment_key=None, inplace=True)

In [ ]:
# plot distribution of perturbations across datasets to check dataset-uniqueness
ct = pd.crosstab(data.obs['pert'], data.obs['dataset'])  # perturbation x dataset counts
datasets_per_pert = (ct > 0).sum(axis=1)

# summary
n_total = len(datasets_per_pert)
n_unique = (datasets_per_pert == 1).sum()
print(f"{n_unique} / {n_total} perturbations ({n_unique/n_total:.1%}) occur in exactly one dataset")

# figure: histogram of how many datasets each perturbation appears in + heatmap of top perturbations
topN = 50
top_pert = ct.sum(axis=1).sort_values(ascending=False).head(topN).index
ct_top = ct.loc[top_pert]
ct_top_frac = ct_top.divide(ct_top.sum(axis=1), axis=0).fillna(0)  # row-normalized fractions

fig, axes = plt.subplots(1, 2, figsize=(18, 10), gridspec_kw={'width_ratios': [1, 1.8]})

# histogram
axes[0].hist(datasets_per_pert, bins=range(1, datasets_per_pert.max() + 2), color='C0', edgecolor='k', align='left')
axes[0].set_xlabel('Number of datasets per perturbation')
axes[0].set_ylabel('Number of perturbations')
axes[0].axvline(1, color='red', linestyle='--', label='unique (1 dataset)')
axes[0].legend()
axes[0].set_title('How many datasets contain each perturbation')

# heatmap for top perturbations (row-normalized)
sns.heatmap(ct_top_frac, ax=axes[1], cmap='viridis', cbar_kws={'label': 'Fraction of cells in dataset'},
            yticklabels=True)
axes[1].set_ylabel('Perturbation (top by count)')
axes[1].set_xlabel('Dataset')
axes[1].set_title(f'Top {topN} perturbations: dataset composition (row-normalized)')

plt.tight_layout()
plt.show()

In [ ]:
# plot number of cells per perturbation: histogram + top N barplot
pert_counts = ct.sum(axis=1)  # ct is perturbation x dataset counts

fig, axes = plt.subplots(1, 2, figsize=(18, 8), gridspec_kw={'width_ratios': [1, 2]})

# histogram of counts
axes[0].hist(pert_counts.values, bins=50, color='C0', edgecolor='k', alpha=0.8)
axes[0].set_xlabel('Cells per perturbation')
axes[0].set_ylabel('Number of perturbations')
axes[0].set_title('Distribution of cells per perturbation')
axes[0].axvline(pert_counts.median(), color='red', linestyle='--', label=f'median={int(pert_counts.median())}')
axes[0].legend()

# horizontal barplot of top N perturbations
topN = min(50, len(pert_counts))
top = pert_counts.sort_values(ascending=False).head(topN)
sns.barplot(x=top.values, y=top.index, palette='viridis', ax=axes[1])
axes[1].set_xlabel('Number of cells')
axes[1].set_ylabel('Perturbation')
axes[1].set_title(f'Top {topN} perturbations by cell count')

plt.tight_layout()
plt.show()

In [ ]:
# Cross-tab of counts: rows=perturbations, cols=datasets
ct = pd.crosstab(data.obs['pert'], data.obs['dataset'])

# Number of perturbations unique to the dataset
datasets_per_pert = (ct > 0).sum(axis=1)              # in how many datasets each pert appears
unique_perts = datasets_per_pert[datasets_per_pert == 1].index
unique_perts_per_dataset = (ct.loc[unique_perts] > 0).sum(axis=0).reindex(ct.columns, fill_value=0)

# Number of total perturbations in the dataset
total_perts_per_dataset = (ct > 0).sum(axis=0).astype(int)

# Number of cells in the dataset
cells_per_dataset = ct.sum(axis=0).astype(int)

# Average number of cells per perturbation in the dataset
avg_cells_per_pert = (cells_per_dataset / total_perts_per_dataset.replace(0, np.nan)).fillna(0).round(2)

# Assemble a single summary table
summary = pd.DataFrame({
    "unique_perturbations": unique_perts_per_dataset.astype(int),
    "total_perturbations": total_perts_per_dataset.astype(int),
    "cells": cells_per_dataset.astype(int),
    "avg_cells_per_perturbation": avg_cells_per_pert
}).reindex(ct.columns)

print("Per-dataset summary:")
print(summary)


In [ ]:
data = data[data.obs.dataset == '241203_kraken']

In [ ]:
# plot number of cells per perturbation: histogram + top N barplot
ct = pd.crosstab(data.obs['pert'], data.obs['dataset'])
pert_counts = ct.sum(axis=1)  # ct is perturbation x dataset counts

fig, axes = plt.subplots(1, 2, figsize=(18, 8), gridspec_kw={'width_ratios': [1, 2]})

# histogram of counts
axes[0].hist(pert_counts.values, bins=50, color='C0', edgecolor='k', alpha=0.8)
axes[0].set_xlabel('Cells per perturbation')
axes[0].set_ylabel('Number of perturbations')
axes[0].set_title('Distribution of cells per perturbation')
axes[0].axvline(pert_counts.median(), color='red', linestyle='--', label=f'median={int(pert_counts.median())}')
axes[0].legend()

# horizontal barplot of top N perturbations
topN = min(50, len(pert_counts))
top = pert_counts.sort_values(ascending=False).head(topN)
sns.barplot(x=top.values, y=top.index, palette='viridis', ax=axes[1])
axes[1].set_xlabel('Number of cells')
axes[1].set_ylabel('Perturbation')
axes[1].set_title(f'Top {topN} perturbations by cell count')

plt.tight_layout()
plt.show()

In [ ]:
min_cells = 20

# Global counts per perturbation (across all datasets)
pert_counts_full = data.obs['pert'].value_counts()

# Keep perturbations with >= min_cells total cells (ignore per-dataset splits)
valid_perts = pert_counts_full[pert_counts_full >= min_cells].index

mask = data.obs['pert'].isin(valid_perts)
print(f"Keeping {mask.sum()} / {data.n_obs} cells ({mask.sum()/data.n_obs:.1%}) "
      f"from {len(valid_perts)} perturbations (>= {min_cells} cells total)")

# Subset
data = data[mask].copy()

# Recompute summaries
ct = pd.crosstab(data.obs['pert'], data.obs['dataset'])
pert_counts = ct.sum(axis=1).astype(int)
datasets_per_pert = (ct > 0).sum(axis=1).astype(int)


In [ ]:
# plot number of cells per perturbation: histogram + top N barplot
ct = pd.crosstab(data.obs['pert'], data.obs['dataset'])
pert_counts = ct.sum(axis=1)  # ct is perturbation x dataset counts

fig, axes = plt.subplots(1, 2, figsize=(18, 8), gridspec_kw={'width_ratios': [1, 2]})

# histogram of counts
axes[0].hist(pert_counts.values, bins=50, color='C0', edgecolor='k', alpha=0.8)
axes[0].set_xlabel('Cells per perturbation')
axes[0].set_ylabel('Number of perturbations')
axes[0].set_title('Distribution of cells per perturbation')
axes[0].axvline(pert_counts.median(), color='red', linestyle='--', label=f'median={int(pert_counts.median())}')
axes[0].legend()

# horizontal barplot of top N perturbations
topN = min(50, len(pert_counts))
top = pert_counts.sort_values(ascending=False).head(topN)
sns.barplot(x=top.values, y=top.index, palette='viridis', ax=axes[1])
axes[1].set_xlabel('Number of cells')
axes[1].set_ylabel('Perturbation')
axes[1].set_title(f'Top {topN} perturbations by cell count')

plt.tight_layout()
plt.show()

In [ ]:
import scipy.sparse as sps

# fallback: sum across expression matrix
lib_size = np.asarray(data.X.sum(axis=1)).ravel()

# 2) number of cells per gene (non-zero counts)
X = data.X
if sps.issparse(X):
    cells_per_gene = np.asarray((X > 0).sum(axis=0)).ravel()
else:
    cells_per_gene = (X > 0).sum(axis=0).astype(int)

# 3) Plotting
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Library size histogram
axes[0].hist(np.log(lib_size+1), bins=100, color='C0', edgecolor='k', alpha=0.8)
axes[0].set_title('Log Library size per cell')
axes[0].set_xlabel('Total counts / cell')
axes[0].set_ylabel('Number of cells')
axes[0].legend()

# Cells per gene histogram
axes[1].hist(cells_per_gene, bins=100, color='C2', edgecolor='k', alpha=0.8)
axes[1].set_title('Number of cells expressing each gene')
axes[1].set_xlabel('Cells with expression > 0')
axes[1].set_ylabel('Number of genes')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# filter cells by log1p(lib_size) > 3 and subset to highly variable genes
cell_mask = np.log(lib_size + 1) > 3
print(f"Keeping {cell_mask.sum()} / {data.n_obs} cells (log1p(lib_size) > 3)")

data = data[cell_mask].copy()

In [ ]:
slk.pp.treat_effect(
    data,
    label_col = 'pert',
    control_label = 'NTC',
    method = "perturbseq",
    inplace=True,
    key='treat_effect'
)

In [ ]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# results = slk.tl.run_single(data, 
#                             batch_size=4096,
#                              shuffle=True, 
#                              num_workers=0, 
#                              device=device, 
#                              ce_lambda=1e-3, 
#                              num_epochs=1000,
#                              l0_lambda=10,
#                              latent_dim=16,
#                              patience=1000,
#                              ntc_lambda=1e-7,
#                              pert_lambda=1e8,
#                              rank=6,
#                              gate_init_p=0.5,
#                              tau_init=0.5,
#                              tau_end=0.1,
#                              shift='linear'
#                              )

In [ ]:
#slk.tl.save_results(results, f'{paths.RESULTS_DIR}/models/raefish.pth')

In [ ]:
# model = slk.tl.load_results(f'{paths.RESULTS_DIR}/models/raefish.pth')

In [ ]:
# NUM_EPOCHS     = 2000
# VALIDATE_EVERY = 50
# BATCH_SIZE     = 4096
# LATENT_DIM     = 20
# PATIENCE       = 350
# SHERLOCK_KWARGS = {
#     # "tau_end": 0.3,
#     # # "tau_init": 1.5,
#     # "cov_lambda": 1e-5,
#     # "H_lambda": 1e-1,
#     # "gate_row_repulsion_lambda": 1e-2,
#     # "ce_lambda": 10,
#     # "shift": "linear",
#     "l0_lambda":35,
#     "debug": True,
#     # "lr": 1e-3,
#     # "ntc_lambda": 1e-5,       
#     # "pert_lambda": 1,
#     "n_epochs_l0_warmup": 400
# }


# seed = 3
# DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# results = slk.tl.run(
#         data,
#         model="vae",
#         batch_size=BATCH_SIZE,
#         num_epochs=NUM_EPOCHS,
#         device=DEVICE,
#         latent_dim=LATENT_DIM,
#         patience=PATIENCE,
#         validate_every=VALIDATE_EVERY,
#         seed=seed,
#         # debug=True,
#         **SHERLOCK_KWARGS,
#     )

In [ ]:
# import datetime
    
# timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# slk.tl.save_results(results, f'{paths.RESULTS_DIR}/models/raefish_model_{timestamp}_{seed}.pth')

# print(f'Saved results to ../data/raefish_model_{timestamp}_{seed}.pth')

In [ ]:
# Saved results to ../data/raefish_model_20260531_202810_1.pth
# Saved results to ../data/raefish_model_20260601_001517_0.pth 
# Saved results to ../data/raefish_model_20260603_110011_2.pth
# Saved results to ../data/raefish_model_20260603_111544_2.pth

In [ ]:
model = slk.tl.load_results(f'{paths.RESULTS_DIR}/models/raefish_model_20260603_112214_3.pth')

In [ ]:
# model = results['model']

In [ ]:
slk.tl.eval_single(model, data, obsm_key="z", uns_key="results")

In [ ]:
data.uns['results']

In [ ]:
slk.pl.plot_r2(data)

In [ ]:
slk.pl.plot_corr(data)

In [ ]:
p_key = slk.configs.get_config('pert_key')
NTC = slk.configs.get_config('ntc_label')

sub = data[data.obs[p_key] != NTC]

sc.pp.neighbors(sub, n_neighbors=15, use_rep='z')
sc.tl.umap(sub)


n_cats = sub.obs[p_key].nunique()
palette = sns.color_palette("hsv", n_colors=n_cats)
random.shuffle(palette)
sc.pl.umap(sub, color='pert', palette=palette, frameon=False)

In [ ]:
import matplotlib as mpl
mpl.rcParams.update({'font.size': 12, 'svg.fonttype': 'none','figure.figsize': (3, 2) }) # Set default figure size to 6×4 inches})
mpl.rcParams['axes.titlesize'] = 16
mpl.rcParams['xtick.labelsize'] = 12
mpl.rcParams['ytick.labelsize'] = 12
mpl.rcParams['axes.labelsize'] = 12
mpl.rcParams['lines.linewidth'] = 2
figure_dir = f"{paths.FIGURES_DIR}/raefish/"
os.makedirs(figure_dir, exist_ok=True)
pert_type = "spatial"

In [ ]:
p_key = slk.configs.get_config('pert_key')
NTC = slk.configs.get_config('ntc_label')

sub = data[data.obs[p_key] != NTC]

sc.pp.neighbors(sub, n_neighbors=15, use_rep='z')
sc.tl.umap(sub)

n_cats = sub.obs[p_key].nunique()
palette = sns.color_palette("hsv", n_colors=n_cats)
random.seed(42)  # for reproducibility
random.shuffle(palette)
plt.rcParams.update({'font.size': 8}) 
sc.pl.umap(sub, color='pert', palette=palette, frameon=False, show=False, s=10)
plt.tight_layout()
plt.savefig(os.path.join(figure_dir, f"{pert_type}_sherlock_zspace_labeled.svg"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{pert_type}_sherlock_zspace_labeled.png"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{pert_type}_sherlock_zspace_labeled.pdf"), dpi=300, bbox_inches='tight')
plt.show()
plt.rcParams.update({'font.size': 12}) 

### Fig. 2a: SHERLOCK latent space, by perturbation

In [ ]:
plt.rcParams.update({'font.size': 8}) 
sc.pl.umap(sub, color='pert', palette=palette, frameon=False, show=False,legend_loc='none', s=10)
plt.tight_layout()
plt.savefig(os.path.join(figure_dir, f"{pert_type}_sherlock_zspace.svg"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{pert_type}_sherlock_zspace.png"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{pert_type}_sherlock_zspace.pdf"), dpi=300, bbox_inches='tight')
plt.show()
plt.rcParams.update({'font.size': 12}) 

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
sc.pl.umap(sub, color='pert', palette=palette, frameon=False, legend_loc='on data', s=20, ax=ax, show=False)
plt.tight_layout()

In [ ]:
zrc = data.uns['results']['z_rho_corr']

print(f"Spearman ρ = {zrc['spearman_r']:.3f}  (p = {zrc['spearman_p']:.1e})")
print(f" Pearson  r = {zrc['pearson_r']:.3f}  (p = {zrc['pearson_p']:.1e})") 

In [ ]:
slk.pl.plot_zcorr(data)

In [ ]:
W = data.uns['results']['W']
plt.hist(W.flatten(), bins=30)
plt.show()

In [ ]:
W_bin = (W > 0.5).astype(float)
sns.heatmap(W, cmap="coolwarm")

In [ ]:

dataset = slk.tl.PerturbMatchingDataset(data)
all_genes = dataset.perturbation_dict.keys()

In [ ]:
groups = slk.tl.cluster_rho(data, t=1.5, show=True)

In [ ]:
#for a non subsetted data, pass in genes corresponding to idx ie dataset.perturbation_dict.keys()
named_groups = slk.tl.group2name(groups, all_genes)
named_groups

In [ ]:
# map perturbation -> group using named_groups (index=gene, column='group')
gene_to_group = named_groups['group'].to_dict()

# assign group to obs based on slk pert key
sub.obs['pert_group'] = sub.obs[p_key].map(gene_to_group)

# mark NTC / unknown perturbations as -1 and ensure integer dtype
sub.obs['pert_group'] = sub.obs['pert_group'].fillna(-1).astype('category')

# quick summary
print(sub.obs['pert_group'].value_counts().sort_index())

sc.pl.umap(sub, color=['pert_group', 'dataset'], title='Perturbation Groups', palette='tab20', size=30)

In [ ]:
new_colormap = [
'darkslateblue', # 0
'firebrick', # 1
'darkorange', # 2
'forestgreen', # 3
'gold', # 4
'dodgerblue', # 5
'mediumorchid', # 6
'deeppink', # 7
'teal', # 8
'black' # 9
]

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster, set_link_color_palette
from matplotlib.colors import to_hex

def cluster_rho(adata, t=1.5, uns_key='results', rho_corr=None, show=True, pert_key=None, figure_dir=None, save_name=None):
    if rho_corr is None:
        C = adata.uns[uns_key]['rho_corr']
    else:
        C = rho_corr
    
    # Get perturbation names - try multiple possible locations
    if 'pert_names' in adata.uns.get(uns_key, {}):
        pert_names = adata.uns[uns_key]['pert_names']
    elif 'rho_perts' in adata.uns.get(uns_key, {}):
        pert_names = list(adata.uns[uns_key]['rho_perts'])
    elif 'perts' in adata.uns.get(uns_key, {}):
        pert_names = adata.uns[uns_key]['perts']
    else:
        # Fall back to unique values from obs
        _pk = pert_key or slk.configs.get_config('pert_key')
        pert_names = adata.obs[_pk].cat.categories.tolist()
        # Remove control if present
        pert_names = [p for p in pert_names if p.lower() not in ['control', 'ctrl', 'non-targeting']]
    
    D = 1.0 - C  # distance
    Z_link = linkage(D[np.triu_indices_from(D, 1)], method='ward')
    
    # Get cluster groups first for coloring
    groups = fcluster(Z_link, t=t, criterion='distance')
    
    if show:
        # Create cluster to color mapping
        unique_groups = sorted(set(groups))
        
        # Set default colors for dendrogram using new_colormap
        palette_hex = [to_hex(c) if isinstance(c, tuple) else c for c in new_colormap[:len(unique_groups)]]
        set_link_color_palette(palette_hex)
        
        plt.figure(figsize=(max(5, len(pert_names) * 0.11), 3.5))
        dendrogram(
            Z_link,
            color_threshold=t,
            above_threshold_color="gray",
            labels=pert_names,  # Use actual names
            leaf_rotation=90,
            leaf_font_size=8,
        )
        plt.axhline(y=t, color="red", linestyle="--", linewidth=1)
        #plt.title(f"Covariance hierarchical clustering t={t}")
        plt.xlabel("Perturbation")
        plt.ylabel("Distance")
        plt.tight_layout()
        
        # Export the figures
        if figure_dir and save_name:
            plt.savefig(os.path.join(figure_dir, f"{save_name}_dendrogram.svg"), dpi=300, bbox_inches='tight')
            plt.savefig(os.path.join(figure_dir, f"{save_name}_dendrogram.png"), dpi=300, bbox_inches='tight')
            plt.savefig(os.path.join(figure_dir, f"{save_name}_dendrogram.pdf"), dpi=300, bbox_inches='tight')
            print(f"Saved figures to {figure_dir}/{save_name}_dendrogram.[svg|png|pdf]")
        
        plt.show()
    
    # Return as a dictionary mapping perturbation names to cluster IDs
    cluster_assignments = dict(zip(pert_names, groups))
    
    return groups, cluster_assignments

### Fig. 2b: hierarchical clustering of perturbations

In [ ]:
groups = cluster_rho(data, t=1.5, show=True, figure_dir=figure_dir, save_name=f"{pert_type}_")

### Fig. 2c: SHERLOCK latent space, by perturbation group

In [ ]:

import matplotlib as mpl
# Update your existing rcParams
mpl.rcParams.update({
    'font.size': 8,
    'svg.fonttype': 'none',
    'font.family': 'Arial',
    'figure.figsize': (3, 2)  # Set default figure size to 6×4 inches
})
mpl.rcParams['axes.titlesize'] = 8
mpl.rcParams['xtick.labelsize'] = 8  # Colorbar tick label size 
mpl.rcParams['ytick.labelsize'] = 8  # Colorbar tick label size

sc.pl.umap(sub, color=['pert_group'], title='', palette=new_colormap, size=15, frameon=False, show=False)

plt.tight_layout()
plt.savefig(os.path.join(figure_dir, f"{pert_type}_sherlock_zspace_conditions_group.svg"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{pert_type}_sherlock_zspace_conditions_group.png"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{pert_type}_sherlock_zspace_conditions_group.pdf"), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy.cluster.hierarchy import linkage, leaves_list
from matplotlib import gridspec
from matplotlib.patches import Patch


def plot_zcorr_colored(adata, uns_key='results', show_pert_labels=True, group_col='pert_group', group_color = new_colormap):
    """
    Plot correlation matrices with color bars for each perturbation group.
    
    Parameters
    ----------
    adata : AnnData
        AnnData object with results in adata.uns[uns_key].
    uns_key : str, default 'results'
        Key in adata.uns where results are stored.
    show_pert_labels : bool, default True
        Whether to show perturbation labels on axes.
    group_col : str, default 'pert_group'
        Column in adata.obs containing perturbation groups.
    """
    
    Sigma_P_posterior = adata.uns[uns_key]['rho_corr']
    Sigma_z = adata.uns[uns_key]['z_corr']
    pert_names = adata.uns[uns_key].get('rho_perts', adata.uns[uns_key].get('perts'))

    link = linkage(Sigma_P_posterior, method="average", metric="euclidean")
    order = leaves_list(link)              # permutation indices, shape (P,)

    # Apply the same permutation to both matrices
    SigmaP_ord = Sigma_P_posterior[order][:, order]
    Sigmaz_ord = Sigma_z[order][:, order]
    
    # Reorder perturbation names
    pert_names_ord = pert_names[order]
    
    # Get perturbation groups
    p_key = slk.configs.get_config('pert_key')
    pert_to_group = adata.obs.groupby(p_key)[group_col].first().to_dict()
    
    # Map perturbations to groups and create colors
    pert_groups = [pert_to_group.get(name, -1) for name in pert_names_ord]
    unique_groups = sorted(set(pert_groups))
    n_groups = len(unique_groups)
    
    # Create color palette for groups (not individual perturbations)
    group_colors = sns.color_palette(group_color, n_groups)
    group_to_color = {grp: group_colors[i] for i, grp in enumerate(unique_groups)}
    row_colors = [group_to_color[grp] for grp in pert_groups]
    
    n_perts = len(pert_names_ord)

    # Thickness controls for color bars
    top_bar_height = 0.8   # was 0.5
    side_bar_width = 0.8   # was 0.5

    # Plot single heatmap
    fig = plt.figure(figsize=(8, 5))
    ax0 = fig.add_subplot(111)
    
    sns.heatmap(SigmaP_ord, cmap="coolwarm", square=True, ax=ax0, 
                xticklabels=pert_names_ord if show_pert_labels else False,
                yticklabels=pert_names_ord if show_pert_labels else False,
                cbar_kws={'label': 'Correlation'})
    ax0.set_title(r"$\Sigma_P$  (posterior row covariance)")
    ax0.set_xlabel("perturbation")
    ax0.set_ylabel("perturbation")
    
    # Make font size smaller
    if show_pert_labels:
        ax0.set_xticklabels(ax0.get_xticklabels(), fontsize=5)
        ax0.set_yticklabels(ax0.get_yticklabels(), fontsize=5)
    
    # Add color bar on top and right (thicker)
    for i, color in enumerate(row_colors):
        # Top
        ax0.add_patch(plt.Rectangle((i, -0.5), 1, top_bar_height, color=color, 
                                    transform=ax0.transData, clip_on=False))
        # Right
        ax0.add_patch(plt.Rectangle((n_perts, i), side_bar_width, 1, color=color, 
                                    transform=ax0.transData, clip_on=False))

    # Create legend outside with transparent background
    legend_elements = [Patch(facecolor=group_to_color[grp], label=f'Group {grp}') 
                      for grp in unique_groups]
    fig.legend(
        handles=legend_elements,
        loc='center left',
        bbox_to_anchor=(1.0, 0.5),
        title='Perturbation Groups',
        fontsize=8,
        frameon=False
    )

    plt.tight_layout()
    plt.savefig(os.path.join(figure_dir, f"{pert_type}_correlation_color.svg"), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(figure_dir, f"{pert_type}_correlation_color.png"), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(figure_dir, f"{pert_type}_correlation_color.pdf"), dpi=300, bbox_inches='tight')
    plt.show()

plot_zcorr_colored(sub)

In [ ]:
for i in sub.obs.pert_group.unique():
    print(i)
    print(sub[sub.obs.pert_group == i].obs['pert'].value_counts())

In [ ]:
# Print gene symbols per group, one gene per line (no quotation marks)
grouped = named_groups.groupby('group').apply(lambda df: df.index.tolist())

for grp in sorted(grouped.index):
    print(f"Group {grp}:")
    for gene in grouped.loc[grp]:
        print(gene)
    print()

In [ ]:
grouped

In [ ]:
# GO enrichment analysis per group using gseapy Enrichr
import gseapy as gp
import pandas as pd
import math
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

GO_LIBS     = ["GO_Biological_Process_2023", "GO_Molecular_Function_2023"]
PVAL_CUTOFF = 0.05
TOP_N       = 10

all_results = {}

for grp in sorted(grouped.index):
    genes = [g for g in grouped.loc[grp] if isinstance(g, str)]
    if len(genes) < 3:
        print(f"Group {grp}: too few genes ({len(genes)}), skipping.")
        continue
    group_dfs = []
    for lib in GO_LIBS:
        try:
            enr = gp.enrichr(gene_list=genes, gene_sets=lib, organism="human",
                             outdir=None, verbose=False)
            df = enr.results.copy()
            df["library"] = lib
            group_dfs.append(df)
        except Exception as e:
            print(f"Group {grp} / {lib}: {e}")
    if group_dfs:
        combined = pd.concat(group_dfs, ignore_index=True)
        combined = combined[combined["Adjusted P-value"] < PVAL_CUTOFF].copy()
        combined["-log10(adjP)"] = -combined["Adjusted P-value"].apply(
            lambda x: math.log10(max(x, 1e-300)))
        all_results[grp] = combined
        print(f"Group {grp} ({len(genes)} genes): {len(combined)} significant GO terms")
    else:
        print(f"Group {grp}: no results")

# Plot top terms per group
groups_with_hits = [g for g in sorted(all_results) if len(all_results[g]) > 0]
if not groups_with_hits:
    print("No significant GO terms found in any group.")
else:
    n_groups = len(groups_with_hits)
    fig, axes = plt.subplots(n_groups, 1,
                             figsize=(12, max(4, n_groups * (TOP_N * 0.25 + 1))))
    if n_groups == 1:
        axes = [axes]
    for ax, grp in zip(axes, groups_with_hits):
        df = all_results[grp].nsmallest(TOP_N, "Adjusted P-value").copy()
        df = df.sort_values("-log10(adjP)", ascending=True)
        colors = ["#4C72B0" if "Biological" in lib else "#DD8452"
                  for lib in df["library"]]
        ax.barh(df["Term"], df["-log10(adjP)"], color=colors)
        ax.axvline(x=-math.log10(PVAL_CUTOFF), color="red", linestyle="--", linewidth=0.8)
        ax.set_title(f"Group {grp}  ({len(grouped.loc[grp])} genes)", fontsize=9)
        ax.set_xlabel("-log10(adj. p-value)", fontsize=8)
        ax.tick_params(axis="y", labelsize=7)
        ax.tick_params(axis="x", labelsize=7)
    legend_elements = [Patch(facecolor="#4C72B0", label="GO Biological Process"),
                       Patch(facecolor="#DD8452", label="GO Molecular Function")]
    fig.legend(handles=legend_elements, loc="upper right", fontsize=8)
    plt.suptitle("GO Enrichment per Group (Enrichr)", y=1.002, fontsize=11)
    plt.tight_layout()
    plt.savefig(f"{figure_dir}/go_enrichment_per_group.png", dpi=300, bbox_inches="tight")
    plt.savefig(f"{figure_dir}/go_enrichment_per_group.pdf", bbox_inches="tight")
    plt.savefig(f"{figure_dir}/go_enrichment_per_group.svg", bbox_inches="tight")
    plt.show()

# Save full results table
go_all = pd.concat(
    [df.assign(group=grp) for grp, df in all_results.items()],
    ignore_index=True
)
go_all.to_csv(f"{figure_dir}/go_enrichment_per_group.csv", index=False)
print(f"Saved full GO results to {figure_dir}/go_enrichment_per_group.csv")


### Fig. 2d: top GO term per perturbation group

In [ ]:
# GO enrichment analysis per group — one top term per group in a single figure
import gseapy as gp
import pandas as pd
import math
import matplotlib.pyplot as plt
from matplotlib.patches import Patch



GO_LIBS     = ["GO_Biological_Process_2023", "GO_Molecular_Function_2023"]
PVAL_CUTOFF = 0.05

top_terms = []

for grp in sorted(grouped.index):
    genes = [g for g in grouped.loc[grp] if isinstance(g, str)]
    if len(genes) < 3:
        continue
    group_dfs = []
    for lib in GO_LIBS:
        try:
            enr = gp.enrichr(gene_list=genes, gene_sets=lib, organism="human",
                             outdir=None, verbose=False)
            df = enr.results.copy()
            df["library"] = lib
            group_dfs.append(df)
        except Exception as e:
            print(f"Group {grp} / {lib}: {e}")
    if not group_dfs:
        continue
    combined = pd.concat(group_dfs, ignore_index=True)
    combined = combined[combined["Adjusted P-value"] < PVAL_CUTOFF]
    if combined.empty:
        print(f"Group {grp}: no significant terms")
        continue
    best = combined.nsmallest(1, "Adjusted P-value").iloc[0]
    # Overlap column is hits/pathway_size — extract hits in group
    overlap_str = best.get("Overlap", "?/?")
    hits = overlap_str.split("/")[0] if "/" in str(overlap_str) else "?"
    top_terms.append({
        "group":      grp,
        "term":       best["Term"],
        "adjP":       best["Adjusted P-value"],
        "-log10P":    -math.log10(max(best["Adjusted P-value"], 1e-300)),
        "library":    best["library"],
        "hits":       hits,
        "group_size": len(genes),
    })

if not top_terms:
    print("No significant GO terms found.")
else:
    # Sort by significance
    summary = pd.DataFrame(top_terms).sort_values("-log10P", ascending=False)
    colors = ["#4C72B0" if "Biological" in lib else "#DD8452"
              for lib in summary["library"]]
    labels = [
        f"Grp {row.group}: {row.term}  ({row.hits}/{row.group_size} genes)"
        for row in summary.itertuples()
    ]

    fig, ax = plt.subplots(figsize=(8, max(2, len(summary) * 0.3)))
    ax.barh(labels, summary["-log10P"], color=colors)
    ax.invert_yaxis()
    ax.axvline(x=-math.log10(PVAL_CUTOFF), color="red", linestyle="--", linewidth=0.8,
               label=f"p={PVAL_CUTOFF}")
    ax.set_xlabel("-log10(adj. p-value)", fontsize=8)
    ax.set_title("Top GO term per group (sorted by significance)", fontsize=8)
    ax.tick_params(axis="y", labelsize=8)
    fig.canvas.draw()
    for lbl in ax.get_yticklabels():
        txt = lbl.get_text()
        try:
            grp_num = int(txt.split(":")[0].replace("Grp", "").strip())
            lbl.set_color(new_colormap[(grp_num - 1) % len(new_colormap)])
        except (ValueError, IndexError):
            pass
    legend_elements = [Patch(facecolor="#4C72B0", label="GO Biological Process"),
                       Patch(facecolor="#DD8452", label="GO Molecular Function")]
    ax.legend(handles=legend_elements, fontsize=8)
    plt.tight_layout()
    plt.savefig(f"{figure_dir}/go_top_term_per_group.png", dpi=300, bbox_inches="tight")
    plt.savefig(f"{figure_dir}/go_top_term_per_group.pdf", bbox_inches="tight")
    plt.savefig(f"{figure_dir}/go_top_term_per_group.svg", bbox_inches="tight")
    plt.show()
    print(summary[["group", "term", "hits", "group_size", "adjP", "library"]].to_string(index=False))


## EGFR inhibitor screen (Giglio et al.)

From [`scripts/egfr_drug_screen/2_egfr_analysis.ipynb`](../scripts/egfr_drug_screen/2_egfr_analysis.ipynb).

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import sys
from pathlib import Path

import seaborn as sns
import matplotlib.pyplot as plt

import torch

import sherlock as slk

In [ ]:
import matplotlib as mpl
mpl.rcParams.update({'font.size': 12, 'svg.fonttype': 'none'})
mpl.rcParams['axes.titlesize'] = 16
mpl.rcParams['xtick.labelsize'] = 12
mpl.rcParams['ytick.labelsize'] = 12
mpl.rcParams['axes.labelsize'] = 12
mpl.rcParams['lines.linewidth'] = 2
figure_dir = f"{paths.FIGURES_DIR}/egfr_drug_screen/"
os.makedirs(figure_dir, exist_ok=True)

In [ ]:
pert_type = 'drug_egfr'  # 'crispri' or 'crisprko'

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
slk.configs.show()
slk.configs.set_config('ntc_label', 'DMSO')
slk.configs.set_config('pert_key', 'drug')
slk.configs.show()
#slk.configs.reset_config()
slk.configs.get_config('ntc_label')


In [ ]:
slk.configs.get_config('ntc_label')

In [ ]:
concentration = 10000
data = sc.read_h5ad(f"{paths.DATA_DIR}/gbm/bt333_egfr_degs_{concentration}.h5ad")

data.obs

In [ ]:
# remove na from drug column
data = data[~data.obs['drug'].isna()].copy()
data = data[
    (data.obs['dose'] == concentration) | (data.obs['drug'] == 'DMSO')
].copy()

In [ ]:
# filter for drugs has more than 100 cells
drug_counts = data.obs['drug'].value_counts()
drugs_to_keep = drug_counts[drug_counts > 100].index
data = data[data.obs['drug'].isin(drugs_to_keep)].copy()

print(f"Kept {len(drugs_to_keep)} drugs with > 100 cells.")
print(data.obs['drug'].value_counts())

In [ ]:
slk.pp.slk_prepare_data(data, pert_key="drug", ntc_label="DMSO", inplace=True)

In [ ]:
slk.pp.treat_effect(
    data,
    label_col = "drug",
    control_label = 'DMSO',
    method = "perturbseq",
    inplace=True,
    key='treat_effect'
)

In [ ]:
data

In [ ]:
data.X.max()

In [ ]:
# Check for NaN/Inf in expression matrix
print(f"Data shape: {data.shape}")
X = data.X.toarray() if hasattr(data.X, 'toarray') else data.X

n_nan = np.isnan(X).sum()
n_inf = np.isinf(X).sum()
print(f"NaN values in X: {n_nan}")
print(f"Inf values in X: {n_inf}")

if n_nan > 0 or n_inf > 0:
    print("Cleaning expression matrix...")
    # Replace NaN with 0 and clip Inf values
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    data.X = X
    print("Expression matrix cleaned!")

# Check data stats
print(f"X min: {data.X.min()}, max: {data.X.max()}")
print(f"Unique drugs: {data.obs['drug'].nunique()}")

In [ ]:
data.obs

In [ ]:
data

In [ ]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# results = slk.tl.run_single(data, 
#                             batch_size=1024,
#                             num_workers=0, 
#                             device=device, 
#                             use_conditions=False,
#                             l0_lambda = 10,
#                             num_epochs=1000,
#                             #gate_init_p=0.4,
#                             H_lambda=0.01,
#                             ce_lambda=0.1,
#                             latent_dim= 8,
#                             )

In [ ]:
# import datetime
    
# timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# slk.tl.save_results(results, f'{paths.RESULTS_DIR}/models/{pert_type}_model_{timestamp}.pth')

In [ ]:
NUM_EPOCHS     = 1000
VALIDATE_EVERY = 50
BATCH_SIZE     = 4096
LATENT_DIM     = 16
PATIENCE       = 350

In [ ]:
SHERLOCK_KWARGS = {
    # "tau_end": 0.3,
    # # "tau_init": 1.5,
    # "cov_lambda": 1e-5,
    # "H_lambda": 1e-1,
    # "gate_row_repulsion_lambda": 1e-2,
    # "ce_lambda": 10,
    # "shift": "linear",
    "l0_lambda":50,
    "debug": True,
    # "lr": 1e-3,
    # "ntc_lambda": 1e-5,       
    # "pert_lambda": 1,
    "n_epochs_l0_warmup": 100
}

In [ ]:
seed = 1

In [ ]:
# DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# results = slk.tl.run(
#         data,
#         model="vae",
#         batch_size=BATCH_SIZE,
#         num_epochs=NUM_EPOCHS,
#         device=DEVICE,
#         latent_dim=LATENT_DIM,
#         patience=PATIENCE,
#         validate_every=VALIDATE_EVERY,
#         seed=seed,
#         # debug=True,
#         **SHERLOCK_KWARGS,
#     )

In [ ]:
# results

In [ ]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# results = slk.tl.run_single(data, batch_size=4096, shuffle=True, num_workers=9, device=device)


In [ ]:
# slk.tl.save_results(results, f'{paths.RESULTS_DIR}/models/replogle_model.pth')


In [ ]:
# import datetime
    
# timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# slk.tl.save_results(results, f'{paths.RESULTS_DIR}/models/{pert_type}_model_{timestamp}_{seed}.pth')

# print(f'Saved results to ../data/{pert_type}_model_{timestamp}_{seed}.pth')

In [ ]:
# model = results['model']

In [ ]:
# This run Ross Saved results to ../data/drug_egfr_model_20260529_163520_1.pth

In [ ]:
# # this is a great run
# #model = slk.tl.load_results(f'{paths.RESULTS_DIR}/models/crispri_model_20251105_103826.pth')
# # {paths.RESULTS_DIR}/models/crispri_model_20251110_215304.pth
# #{paths.RESULTS_DIR}/models/crispri_model_20251111_004834.pth


# # accuray is not 100%
model = slk.tl.load_results(f'{paths.RESULTS_DIR}/models/drug_egfr_model_20260529_163520_1.pth')
# #model = results['model']

In [ ]:
slk.tl.eval_single(model, data, obsm_key="z", uns_key="results")

In [ ]:
data.uns['results']

In [ ]:
data.X = data.X.toarray()
slk.pl.plot_r2(data)

In [ ]:
slk.pl.plot_corr(data)

In [ ]:
p_key = slk.configs.get_config('pert_key')
NTC = slk.configs.get_config('ntc_label')

sub = data[data.obs[p_key] != NTC]

sc.pp.neighbors(sub, n_neighbors=15, use_rep='z')
sc.tl.umap(sub)

import random
n_cats = sub.obs[p_key].nunique()
palette = sns.color_palette("hsv", n_colors=n_cats)
random.shuffle(palette)
sc.pl.umap(sub, color='drug', palette=palette, frameon=False)

In [ ]:
zrc = data.uns['results']['z_rho_corr']

print(f"Spearman ρ = {zrc['spearman_r']:.3f}  (p = {zrc['spearman_p']:.1e})")
print(f" Pearson  r = {zrc['pearson_r']:.3f}  (p = {zrc['pearson_p']:.1e})") 

In [ ]:
slk.pl.plot_zcorr(data)

In [ ]:
W = data.uns['results']['W']
plt.hist(W.flatten(), bins=30)
plt.show()

In [ ]:
W_bin = (W > 0.5).astype(float)
sns.heatmap(W, cmap="coolwarm")

In [ ]:

dataset = slk.tl.PerturbMatchingDataset(data)
all_genes = dataset.perturbation_dict.keys()

In [ ]:

dataset = slk.tl.PerturbMatchingDataset(data)
all_genes = dataset.perturbation_dict.keys()

In [ ]:
new_colormap = [
    'darkslateblue',  # 0
    'firebrick',      # 1
    'darkorange',     # 2
    'forestgreen',    # 3
    'gold',           # 4
    'dodgerblue',     # 5
    'mediumorchid',   # 6
    'deeppink',       # 7
    'teal',           # 8
    'black'           # 9
]
t = 2.2

In [ ]:
# 1.5 previously
groups = slk.tl.cluster_rho(data, t=t
                            , show=True)

In [ ]:
#for a non subsetted data, pass in genes corresponding to idx ie dataset.perturbation_dict.keys()
named_groups = slk.tl.group2name(groups, all_genes)
named_groups

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster, set_link_color_palette
from matplotlib.colors import to_hex

def cluster_rho(adata, t=1.5, uns_key='results', rho_corr=None, show=True, pert_key='perturbation', figure_dir=None, save_name=None):
    if rho_corr is None:
        C = adata.uns[uns_key]['rho_corr']
    else:
        C = rho_corr
    
    # Get perturbation names - try multiple possible locations
    uns = adata.uns.get(uns_key, {})
    if 'rho_perts' in uns:
        pert_names = np.asarray(uns['rho_perts'])
    elif 'pert_names' in uns:
        pert_names = np.asarray(uns['pert_names'])
    elif 'perts' in uns:
        pert_names = np.asarray(uns['perts'])
    else:
        # Fall back to unique perturbation values from obs
        _pk = slk.configs.get_config('pert_key')
        ntc = slk.configs.get_config('ntc_label')
        pert_names = np.asarray([p for p in adata.obs[_pk].unique() if p != ntc])
    
    D = 1.0 - C  # distance
    Z_link = linkage(D[np.triu_indices_from(D, 1)], method='ward')
    
    # Cluster labels at this cut, and the exact colors scipy will draw for them —
    # so plot_zcorr_colored (same linkage, same t, same palette) matches this dendrogram
    groups = fcluster(Z_link, t=t, criterion='distance')
    unique_groups = sorted(set(groups))
    palette_hex = [to_hex(new_colormap[i % len(new_colormap)]) for i in range(len(unique_groups))]
    set_link_color_palette(palette_hex)
    
    if show:
        plt.figure(figsize=(max(5, len(pert_names) * 0.15), 4))
        dendrogram(
            Z_link,
            color_threshold=t,
            above_threshold_color="gray",
            labels=pert_names,  # Use actual names
            leaf_rotation=90,
            leaf_font_size=8,
        )
        plt.axhline(y=t, color="red", linestyle="--", linewidth=1)
        #plt.title(f"Covariance hierarchical clustering t={t}")
        plt.xlabel("Perturbation")
        plt.ylabel("Distance")
        plt.tight_layout()
        
        # Export the figures
        if figure_dir and save_name:
            plt.savefig(os.path.join(figure_dir, f"{save_name}_dendrogram.svg"), dpi=300, bbox_inches='tight')
            plt.savefig(os.path.join(figure_dir, f"{save_name}_dendrogram.png"), dpi=300, bbox_inches='tight')
            plt.savefig(os.path.join(figure_dir, f"{save_name}_dendrogram.pdf"), dpi=300, bbox_inches='tight')
            print(f"Saved figures to {figure_dir}/{save_name}_dendrogram.[svg|png|pdf]")
        
        plt.show()
    
    # Record the group -> color mapping scipy actually used, for reference/reuse
    ddata = dendrogram(Z_link, color_threshold=t, above_threshold_color='gray', no_plot=True)
    group_to_color = {}
    for leaf_idx, color in zip(ddata['leaves'], ddata.get('leaves_color_list', [])):
        g = groups[leaf_idx]
        if g not in group_to_color:
            group_to_color[g] = color
    adata.uns['_dendrogram_colors'] = {str(k): v for k, v in group_to_color.items()}
    
    # Return as a dictionary mapping perturbation names to cluster IDs
    cluster_assignments = dict(zip(pert_names, groups))
    
    return groups, cluster_assignments

### Fig. 2h: hierarchical clustering of drugs

In [ ]:
groups = cluster_rho(data, t=t, show=True, figure_dir=figure_dir, save_name=f"{pert_type}_{concentration}uM")

In [ ]:
#named_groups.to_csv(f'{paths.RESULTS_DIR}/gbm/{pert_type}_drug_groups_{concentration}uM_final.csv')

In [ ]:
# map perturbation -> group using named_groups (index=gene, column='group')
gene_to_group = named_groups['group'].to_dict()

# assign group to obs based on slk pert key
sub.obs['pert_group'] = sub.obs[p_key].map(gene_to_group)

# mark NTC / unknown perturbations as -1 and ensure integer dtype
sub.obs['pert_group'] = sub.obs['pert_group'].fillna(-1).astype('category')

# quick summary
print(sub.obs['pert_group'].value_counts().sort_index())

sc.pl.umap(sub, color=['pert_group'], title='Perturbation Groups', palette='tab20', size=30)

In [ ]:
new_colormap = [
    'darkslateblue',  # 0
    'firebrick',      # 1
    'darkorange',     # 2
    'forestgreen',    # 3
    'gold',           # 4
    'dodgerblue',     # 5
    'mediumorchid',   # 6
    'deeppink',       # 7
    'teal',           # 8
    'black'           # 9
]

### Fig. 2e–f: SHERLOCK latent space, by drug and by perturbation group

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

sc.pl.umap(
    sub,
    color='drug',
    title='Drugs',
    palette=palette,
    size=12,
    frameon=False,
    ax=axes[0],
    show=False,
    legend_fontsize=6,
)

sc.pl.umap(sub, color='pert_group', title='Perturbation groups', 
           palette=new_colormap, size=12, frameon=False, ax=axes[1], show=False)

for ax in axes:
    ax.set_title(ax.get_title(), fontsize=12)

plt.tight_layout()
plt.savefig(os.path.join(figure_dir, f"{pert_type}_{concentration}_sherlock_zspace.svg"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{pert_type}_{concentration}_sherlock_zspace.png"), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figure_dir, f"{pert_type}_{concentration}_sherlock_zspace.pdf"), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# assign group to obs based on slk pert key
data.obs['pert_group'] = data.obs[p_key].map(gene_to_group)

# mark NTC / unknown perturbations as -1 and ensure integer dtype
data.obs['pert_group'] = data.obs['pert_group'].fillna(-1).astype('category')

# quick summary
print(data.obs['pert_group'].value_counts().sort_index())

### Fig. 2g: learned perturbation correlation

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy.cluster.hierarchy import linkage, leaves_list, fcluster, dendrogram, set_link_color_palette
from matplotlib.colors import to_hex
from matplotlib import gridspec
from matplotlib.patches import Patch


def plot_zcorr_colored(adata, t=1.5, uns_key='results', group_color=new_colormap):
    """
    Plot correlation matrices with a color bar for each perturbation group.
    Uses the same linkage/threshold/palette as cluster_rho, so the bar colors
    match what cluster_rho's dendrogram draws for the same cut t.
    """
    
    Sigma_P_posterior = adata.uns[uns_key]['rho_corr']
    Sigma_z = adata.uns[uns_key]['z_corr']
    pert_names = adata.uns[uns_key]['rho_perts']

    # Same distance + linkage as cluster_rho
    D = 1.0 - Sigma_P_posterior
    Z_link = linkage(D[np.triu_indices_from(D, 1)], method='ward')
    order = leaves_list(Z_link)

    # Apply the same permutation to both matrices
    SigmaP_ord = Sigma_P_posterior[order][:, order]
    Sigmaz_ord = Sigma_z[order][:, order]
    pert_names_ord = pert_names[order]

    # Same cut as cluster_rho(..., t=t)
    groups = fcluster(Z_link, t=t, criterion='distance')
    groups_ord = groups[order]
    unique_groups = sorted(set(groups))

    # Same palette + same scipy color-assignment logic as cluster_rho, so colors match the dendrogram
    palette_hex = [to_hex(group_color[i % len(group_color)]) for i in range(len(unique_groups))]
    set_link_color_palette(palette_hex)
    ddata = dendrogram(Z_link, color_threshold=t, above_threshold_color='gray', no_plot=True)
    group_to_color = {}
    for leaf_idx, color in zip(ddata['leaves'], ddata.get('leaves_color_list', [])):
        g = groups[leaf_idx]
        if g not in group_to_color:
            group_to_color[g] = color

    row_colors = [group_to_color[grp] for grp in groups_ord]
    n_perts = len(pert_names_ord)

    # Bar width and its gap from the heatmap, scaled to stay proportional to heatmap
    # size (reference notebook used width=3, gap=1 for 71 perturbations)
    side_bar_width = n_perts * (3 / 71)
    bar_gap = n_perts * (1 / 71)

    # Plot single heatmap
    fig = plt.figure(figsize=(4, 3))
    ax0 = fig.add_subplot(111)
    
    sns.heatmap(SigmaP_ord, cmap="coolwarm", square=True, ax=ax0, 
                xticklabels=False,
                yticklabels=False,
                cbar_kws={
                    'label': 'Correlation',
                    'shrink': 0.6,
                    'aspect': 20
                })
    ax0.set_title("")
    ax0.set_xlabel("Perturbations")
    ax0.set_ylabel("Perturbations")
    
    # Add vertical color bar on the right, with a small gap from the heatmap
    for i, color in enumerate(row_colors):
        ax0.add_patch(plt.Rectangle((n_perts + bar_gap, i), side_bar_width, 1, color=color, 
                                    transform=ax0.transData, clip_on=False))

    # Create legend outside with transparent background
    legend_elements = [Patch(facecolor=group_to_color[grp], label=f'{int(grp)}')
                      for grp in unique_groups]
    fig.legend(
        handles=legend_elements,
        loc='center left',
        bbox_to_anchor=(1.0, 0.5),
        title='Perturbation Groups',
        fontsize=8,
        handlelength=1.5,
        handleheight=1.5,
        frameon=False
    )

    plt.tight_layout()
    plt.savefig(os.path.join(figure_dir, f"{pert_type}_correlation_color.svg"), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(figure_dir, f"{pert_type}_correlation_color.png"), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(figure_dir, f"{pert_type}_correlation_color.pdf"), dpi=300, bbox_inches='tight')
    plt.show()

plot_zcorr_colored(data, t=t)

## Fig. 2i–j: CRISPRi kinome screen

The correlation matrix and dendrogram are drawn from the best of the five CRISPRi runs saved by [`scripts/kinome_crispri_crispra/2a_reproducibility_crispri.ipynb`](../scripts/kinome_crispri_crispra/2a_reproducibility_crispri.ipynb) (`matrices/best_model.pth`), with `slk.pl.plot_corr` and `slk.tl.cluster_rho`. The plotting cells for these two panels are not yet part of that notebook.